# Harmonized Landsat Sentinel 2 (HLS): tiling configuration and rendering performance

## Background

The Harmonized Landsat Sentinel (HLS) dataset provides surface reflectance data by combining observations from the Operational Land Imager (OLI) and Multi-Spectral Instrument (MSI) aboard the Landsat 8/9 and Sentinel-2 satellites, respectively. This dataset is processed to enable time series analyses with greater density of observations in time than is possible with any single global satellite data collection with similar spatial resolution.

Two HLS collections are available in CMR:
- **HLSL30**: Harmonized Landsat 8/9 data (30m resolution)
- **HLSS30**: Harmonized Sentinel-2 data (30m resolution)

These collections have specific characteristics that impact the performance of tile rendering:
1. Each granule covers one MGRS (Military Grid Reference System) tile
2. The data is stored as separate files per band (e.g., B01, B02, etc.)
3. Temporal coverage is not uniform across the globe

## Performance Factors

Several factors affect tiling performance (in order of importance):
1. **Zoom level**: Lower zoom levels require rendering pixels from more assets
2. **Temporal range**: Larger time ranges require searching through and rendering more granules
3. **Collection choice**: HLSS30 has a more frequent revisit rate than HLSL30
    - HLSS30's higher revisit rate means that for a fixed area and datetime interval there will usually be more granules to process than for HLSL30
4. **Band combination**: More bands require more files to be accessed and processed

## Tile rendering benchmark analysis
To assess tile rendering performance for the HLS collections we wrote a [benchmarking process](https://github.com/developmentseed/titiler-cmr/blob/develop/tests/test_hls_benchmark.py) that measures the time it takes for the API to render tiles for a range of zoom levels, datetime interval widths, and number of band combinations.
This notebook explores the trends and findings from the [benchmark results](https://github.com/developmentseed/titiler-cmr/blob/develop/benchmark.json).
The following diagram shows the observed performance across all of these dimensions for the two collections (HLSL30 and HLSS30).
Based on this analysis we recommend these configuration limits:

- **`minzoom=7`**: the titiler-cmr API should be able to return a full viewport's worth of tiles in ~10 seconds.
- use single-day `datetime` intervals to reduce the number of assets requested in a single tile (e.g. `datetime=2024-08-10T00:00:01Z/2024-08-10T23:59:59`)
  - this will give users an orbit-wise view of the data, which is probably the most intuitive way to view it
  - wider datetime intervals will work, but performance can substantially slower
- limiting the other factors do not seem to introduce enough variance in response time to warrant additional guidelines.

In [ ]:
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the benchmark data
with open("../../benchmark.json", "r") as f:
    benchmark_data = json.load(f)

# Set style for plots
plt.style.use("ggplot")
sns.set_palette("colorblind")

results = []
for benchmark in benchmark_data["benchmarks"]:
    # Only include results from the hls-tiles group
    if benchmark["group"] == "hls-tiles":
        row = {
            "name": benchmark["name"],
            "collection": benchmark["extra_info"]["collection"],
            "zoom": benchmark["extra_info"]["zoom"],
            "interval_days": benchmark["extra_info"]["interval_days"],
            "band_count": benchmark["extra_info"]["band_count"],
            "mean_time": benchmark["stats"]["mean"],  # Mean execution time in seconds
            "success_rate": benchmark["extra_info"]["success_rate"],
            "avg_response_time": benchmark["extra_info"]["avg_response_time"],
            "avg_response_size": benchmark["extra_info"]["avg_response_size"],
        }
        results.append(row)

df = pd.DataFrame(results)

collections = df["collection"].unique()
intervals = df["interval_days"].unique()

fig, axes = plt.subplots(
    nrows=len(intervals),
    ncols=len(collections),
    figsize=(14, 10),
    sharex=True,
    sharey=True,
)

band_colors = {1: "blue", 2: "green", 3: "red"}

for i, interval in enumerate(sorted(intervals)):
    for j, collection in enumerate(sorted(collections)):
        ax = axes[i, j]

        facet_data = df[
            (df["collection"] == collection) & (df["interval_days"] == interval)
        ]

        for band_count, band_data in facet_data.groupby("band_count"):
            # Sort by zoom level to ensure proper line connections
            band_data = band_data.sort_values("zoom")

            ax.plot(
                band_data["zoom"],
                band_data["mean_time"],
                marker="o",
                markersize=8,
                color=band_colors[band_count],
                linewidth=2,
                label=f"{band_count} bands",
            )

        # Add facet title only on the first row and column
        if i == 0:
            ax.set_title(collection, fontsize=14)
        if j == 0:
            ax.set_ylabel(f"{interval} day interval\nResponse Time (s)", fontsize=12)

        # Add x-label only on the bottom row
        if i == len(intervals) - 1:
            ax.set_xlabel("zoom Level", fontsize=12)

        # Set x-ticks to show the actual zoom levels
        ax.set_xticks([6, 8, 10])

        # Add grid
        ax.grid(True, alpha=0.3)

        # Only add legend to the first plot
        if i == 0 and j == 0:
            # Create legend for band counts
            band_handles = [
                plt.Line2D(
                    [0],
                    [0],
                    color=band_colors[b],
                    label=f"{b} band{'s' if b > 1 else ''}",
                    marker="o",
                    markersize=6,
                    linewidth=2,
                )
                for b in sorted(band_colors.keys())
            ]
            ax.legend(handles=band_handles, loc="upper left")

# Add an overall title
fig.suptitle(
    "HLS tiling benchmark: time to render tiles for viewport",
    fontsize=16,
    y=0.98,
)

# Adjust layout
plt.tight_layout()
fig.subplots_adjust(top=0.9)

## HLSL30 (Landsat) Parameters

### Basic Parameters
| Parameter | Value |
|-----------|-------|
| `concept_id` | `"C2021957657-LPCLOUD"` |
| `backend` | `"rasterio"` |
| `bands_regex` | `"B[0-9][0-9]"` |
| `minzoom` | `8` |
| `maxzoom` | `13` |

### True Color RGB
| Parameter | Value |
|-----------|-------|
| `bands` | `["B04", "B03", "B02"]` |
| `color_formula` | `"Gamma RGB 3.5 Saturation 1.2 Sigmoidal RGB 15 0.35"` |

### False Color NIR
| Parameter | Value |
|-----------|-------|
| `bands` | `["B05", "B03", "B02"]` |
| `color_formula` | `"Gamma RGB 2.5 Saturation 1.2 Sigmoidal RGB 10 0.35"` |

### NDVI
| Parameter | Value |
|-----------|-------|
| `bands` | `["B05", "B04"]` |
| `expression` | `"(B05-B04)/(B05+B04)"` |
| `colormap_name` | `"summer_r"` |
| `rescale` | `"-0.5,1"` |

## HLSS30 (Sentinel-2) Parameters

### Basic Parameters
| Parameter | Value |
|-----------|-------|
| `concept_id` | `"C2021957295-LPCLOUD"` |
| `backend` | `"rasterio"` |
| `bands_regex` | `"B[0-9][0-9A-Za-z]"` |
| `minzoom` | `8` |
| `maxzoom` | `13` |

### True Color RGB
| Parameter | Value |
|-----------|-------|
| `bands` | `["B04", "B03", "B02"]` |
| `color_formula` | `"Gamma RGB 3.5 Saturation 1.2 Sigmoidal RGB 15 0.35"` |

### False Color NIR
| Parameter | Value |
|-----------|-------|
| `bands` | `["B8A", "B03", "B02"]` |
| `color_formula` | `"Gamma RGB 2.5 Saturation 1.2 Sigmoidal RGB 10 0.35"` |

### NDVI
| Parameter | Value |
|-----------|-------|
| `bands` | `["B8A", "B04"]` |
| `expression` | `"(B8A-B04)/(B8A+B04)"` |
| `colormap_name` | `"summer_r"` |
| `rescale` | `"-0.5,1"` |

## Interactive map examples

This section demonstrates how you might parameterize tilejson requests (in Python) for each of the collections for variety of layers (true color, false color, NDVI).

In [ ]:
import httpx
from folium import LayerControl, Map, TileLayer

titiler_endpoint = "https://dev-titiler-cmr.delta-backend.com"
minzoom = 7

### HLSL30

In [ ]:
m = Map(location=(46.7653, -91.0321), zoom_start=minzoom + 1)
base_parameters = (
    ("concept_id", "C2021957657-LPCLOUD"),
    # Datetime in form of `start_date/end_date`
    # ("datetime", "2025-03-16T00:00:00Z/2025-03-16T23:59:59Z"),  # leaf-off
    ("datetime", "2024-08-06T00:00:00Z/2024-08-06T23:59:59Z"),  # leaf-on
    # We know that the HLS collection dataset is stored as File per Band
    # so we need to pass a `band_regex` option to assign `bands` to each URL
    ("bands_regex", "B[0-9][0-9]"),
    # titiler-cmr can work with both Zarr and COG dataset
    # but we need to tell the endpoints in advance which backend
    # to use
    ("backend", "rasterio"),
    # We need to set min/max zoom because we don't want to use lowerzoom level (e.g 0)
    # which will results in useless large scale query
    ("minzoom", 7),
    ("maxzoom", 13),
)

landsat_true_color_tilejson = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        *base_parameters,
        # True Color Image B04,B03,B02
        ("bands", "B04"),
        ("bands", "B03"),
        ("bands", "B02"),
        # The data is in type of Uint16 so we need to apply some
        # rescaling/color_formula in order to create PNGs
        ("color_formula", "Gamma RGB 3.5 Saturation 1.2 Sigmoidal RGB 15 0.35"),
    ),
    timeout=None,
).json()

landsat_true_color_tile_url = landsat_true_color_tilejson["tiles"][0]


TileLayer(
    tiles=landsat_true_color_tile_url,
    opacity=1,
    name="HLSL30 true color",
    overlay=True,
    show=True,
    control=True,
    attr="NASA",
).add_to(m)

landsat_false_color_tilejson = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        *base_parameters,
        # False Color Image B05,B03,B02
        ("bands", "B05"),
        ("bands", "B03"),
        ("bands", "B02"),
        # The data is in type of Uint16 so we need to apply some
        # rescaling/color_formula in order to create PNGs
        ("color_formula", "Gamma RGB 2.5 Saturation 1.2 Sigmoidal RGB 10 0.35"),
    ),
    timeout=None,
).json()

landsat_false_color_tile_url = landsat_false_color_tilejson["tiles"][0]

TileLayer(
    tiles=landsat_false_color_tile_url,
    opacity=1,
    name="HLSL30 false color",
    overlay=True,
    show=False,
    control=True,
    attr="NASA",
).add_to(m)

landsat_ndvi_tilejson = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        *base_parameters,
        # NDVI: B05 (NIR), B04 (Red)
        ("bands", "B05"),
        ("bands", "B04"),
        # Calculate NDVI from the NIR and Red bands
        ("expression", "(B05-B04)/(B05+B04)"),
        # Rescale to NDVI scale
        ("rescale", "-0.5,1"),
        # Choose a colormap
        ("colormap_name", "summer_r"),
    ),
    timeout=None,
).json()

landsat_ndvi_tile_url = landsat_ndvi_tilejson["tiles"][0]

TileLayer(
    tiles=landsat_ndvi_tile_url,
    opacity=1,
    name="HLSL30 NDVI",
    overlay=True,
    show=False,
    control=True,
    attr="NASA",
).add_to(m)

LayerControl(position="topright", collapsed=False).add_to(m)

m

### HLSS30

In [ ]:
m = Map(location=(46.7653, -91.0321), zoom_start=minzoom + 1)

base_parameters = (
    ("concept_id", "C2021957295-LPCLOUD"),
    # Datetime in form of `start_date/end_date`
    ("datetime", "2024-08-13T00:00:00Z/2024-08-13T23:59:59Z"),  # leaf-on
    # ("datetime", "2025-03-16T00:00:00Z/2025-03-16T23:59:59Z"),  # leaf-off
    # We know that the HLS collection dataset is stored as File per Band
    # so we need to pass a `band_regex` option to assign `bands` to each URL
    ("bands_regex", "B[0-9][0-9A-Za-z]"),
    # titiler-cmr can work with both Zarr and COG dataset
    # but we need to tell the endpoints in advance which backend
    # to use
    ("backend", "rasterio"),
    # We need to set min/max zoom because we don't want to use lowerzoom level (e.g 0)
    # which will results in useless large scale query
    ("minzoom", 7),
    ("maxzoom", 13),
)

sentinel_true_color_tilejson = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        *base_parameters,
        # True Color Image B04,B03,B02
        ("bands", "B04"),
        ("bands", "B03"),
        ("bands", "B02"),
        # The data is in type of Uint16 so we need to apply some
        # rescaling/color_formula in order to create PNGs
        ("color_formula", "Gamma RGB 3.5 Saturation 1.2 Sigmoidal RGB 15 0.35"),
    ),
    timeout=None,
).json()

sentinel_true_color_tile_url = sentinel_true_color_tilejson["tiles"][0]


TileLayer(
    tiles=sentinel_true_color_tile_url,
    opacity=1,
    name="HLSS30 true color",
    overlay=True,
    show=True,
    control=True,
    attr="NASA",
).add_to(m)

sentinel_false_color_tilejson = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        *base_parameters,
        # False Color Image B8A, B03, B02
        ("bands", "B8A"),
        ("bands", "B03"),
        ("bands", "B02"),
        # The data is in type of Uint16 so we need to apply some
        # rescaling/color_formula in order to create PNGs
        ("color_formula", "Gamma RGB 2.5 Saturation 1.2 Sigmoidal RGB 10 0.35"),
    ),
    timeout=None,
).json()

sentinel_false_color_tile_url = sentinel_false_color_tilejson["tiles"][0]

TileLayer(
    tiles=sentinel_false_color_tile_url,
    opacity=1,
    name="HLSS30 false color",
    overlay=True,
    show=False,
    control=True,
    attr="NASA",
).add_to(m)

sentinel_ndvi_tilejson = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        *base_parameters,
        # NDVI: B8A (NIR), B04 (Red)
        ("bands", "B8A"),
        ("bands", "B04"),
        # Calculate NDVI from the NIR and Red bands
        ("expression", "(B8A-B04)/(B8A+B04)"),
        # Rescale to -1,1
        ("rescale", "-0.5,1"),
        # Choose a colormap
        ("colormap_name", "summer_r"),
    ),
    timeout=None,
).json()

sentinel_ndvi_tile_url = sentinel_ndvi_tilejson["tiles"][0]

TileLayer(
    tiles=sentinel_ndvi_tile_url,
    opacity=1,
    name="HLSS30 NDVI",
    overlay=True,
    show=False,
    control=True,
    attr="NASA",
).add_to(m)

LayerControl(position="topright", collapsed=False).add_to(m)

m